# Chapter 10: Speculative Decoding & Advanced Inference

**Goal:** Break the memory-bandwidth barrier for autoregressive decoding by generating multiple tokens per forward pass.

Recall from Chapters 1-2: decode is memory-bound at batch=1. Each token requires loading ALL model weights from HBM. Even with quantization (Ch7), Flash Attention (Ch4), kernel fusion (Ch6), and continuous batching (Ch9), we're still fundamentally limited by memory bandwidth for low-latency single-request inference.

Can we generate **multiple tokens per forward pass**? Yes — with speculative decoding.

**Approach:** Fill in every `# YOUR CODE` section yourself — that's the learning.

**Model:** LLaMA-3.2-1B (d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers)

**Hardware:** A100 80GB SXM4
- Measured HBM bandwidth: 1774 GB/s
- FP16 TFLOPS: 237
- Ridge point: ~134 FLOPS/byte

## 1. The Decode Speed Problem Revisited

During autoregressive decode at batch=1:
- Each token loads ALL model weights: ~2.5 GB for LLaMA-3.2-1B FP16
- At 1774 GB/s bandwidth, minimum time per token = 2.5 GB / 1774 GB/s = ~1.4 ms
- That's ~700 tokens/sec theoretical max — and only ~150 tok/s in practice

The GPU computes for ~1% of peak FLOPS during decode. The other 99% is waiting for memory.

Previous optimizations attack this from different angles:
- **Quantization** (Ch7): Shrink weights -> fewer bytes to load
- **Batching** (Ch9): Amortize weight loading across requests -> higher throughput
- **Flash Attention** (Ch4): Reduce HBM traffic for attention

But what if we could generate **multiple tokens** while loading the weights just **once**?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model params (LLaMA-3.2-1B)
d_model = 2048
n_heads = 32
n_kv_heads = 8
d_head = 64
d_ff = 8192
n_layers = 16

# A100 specs
hbm_bandwidth = 1774  # GB/s
peak_tflops = 237     # FP16 TFLOPS
ridge_point = peak_tflops * 1000 / hbm_bandwidth  # ~134 FLOPS/byte

# Model weight size (FP16)
params_per_layer = (d_model * d_model * 2          # Q + O
                    + d_model * (n_kv_heads * d_head) * 2  # K + V
                    + d_model * d_ff * 3)           # gate + up + down
total_params = params_per_layer * n_layers
weight_bytes = total_params * 2  # FP16

# Time per token at batch=1
time_per_token_ms = weight_bytes / (hbm_bandwidth * 1e9) * 1000
theoretical_max_tps = 1000 / time_per_token_ms

print(f"Model weights: {weight_bytes / 1e9:.2f} GB")
print(f"Minimum time per token (memory-bound): {time_per_token_ms:.2f} ms")
print(f"Theoretical max tokens/sec at batch=1: {theoretical_max_tps:.0f}")
print(f"Practical tokens/sec at batch=1: ~150")
print(f"\nThe problem: each token costs one full weight load.")
print(f"If we could get K tokens per weight load, we'd get K x speed.")

## 2. Speculative Decoding: The Core Idea

The key insight: **verification is cheaper than generation**.

1. Use a **small draft model** to generate K candidate tokens cheaply (K small forward passes)
2. Feed all K candidates into the **large target model** in ONE forward pass (like prefill — compute-bound)
3. **Accept** tokens where draft matches target, **reject** at first divergence
4. Get 1 to K+1 tokens from one target forward pass

**Critical guarantee:** The accepted tokens are distributed **exactly** as if generated by the target model alone. No quality loss whatsoever — this is provable.

Why verification is cheaper:
- Generation: batch=1, memory-bound (load weights per token)
- Verification: batch=K, more compute-per-byte (closer to prefill)

In [ ]:
# --- YOUR CODE: implement the speculative decoding accept/reject logic ---

def speculative_accept_reject(draft_probs, target_probs, draft_tokens):
    """
    Given draft model probabilities, target model probabilities, and draft tokens,
    determine which tokens to accept.

    Args:
        draft_probs: list of probability distributions from draft model (K distributions)
        target_probs: list of probability distributions from target model (K+1 distributions)
        draft_tokens: list of K tokens sampled from draft model

    Returns:
        accepted_tokens: list of accepted tokens (1 to K+1 tokens)
    """
    accepted = []
    K = len(draft_tokens)

    for i in range(K):
        token = draft_tokens[i]
        # --- YOUR CODE: acceptance criterion ---
        # Hint: accept with probability min(1, target_probs[i][token] / draft_probs[i][token])
        p_target = target_probs[i][token]
        p_draft = draft_probs[i][token]
        accept_prob = None  # YOUR CODE: min(1, p_target / p_draft)

        if np.random.random() < accept_prob:
            accepted.append(token)
        else:
            # Reject: sample from adjusted distribution
            # --- YOUR CODE: compute adjusted distribution ---
            # Hint: adjusted = max(0, target - draft) for each token, then normalize
            adjusted = np.maximum(0, target_probs[i] - draft_probs[i])  # YOUR CODE
            adjusted = adjusted / adjusted.sum()  # YOUR CODE: normalize
            resampled = np.random.choice(len(adjusted), p=adjusted)
            accepted.append(resampled)
            return accepted  # Stop at first rejection

    # All K tokens accepted! Sample one more from target's K+1 distribution
    bonus_token = np.random.choice(len(target_probs[K]), p=target_probs[K])
    accepted.append(bonus_token)
    return accepted


# Demonstrate with a simple vocabulary
vocab_size = 10
K = 5  # draft tokens
np.random.seed(42)

# Simulate draft and target distributions
# When draft is close to target -> high acceptance rate
base_dist = np.random.dirichlet(np.ones(vocab_size) * 2)
draft_distributions = [np.random.dirichlet(base_dist * 50) for _ in range(K)]
target_distributions = [np.random.dirichlet(base_dist * 50) for _ in range(K + 1)]

# Sample draft tokens
draft_tokens = [np.random.choice(vocab_size, p=d) for d in draft_distributions]

print(f"Draft model generated {K} candidate tokens: {draft_tokens}")
accepted = speculative_accept_reject(draft_distributions, target_distributions, draft_tokens)
print(f"Accepted tokens: {accepted}")
print(f"Accepted {len(accepted)} tokens from 1 target forward pass")
print(f"\nKey property: accepted tokens are EXACTLY distributed as target model.")
print(f"No quality loss — this is mathematically guaranteed.")

# Run many trials to show average acceptance
n_trials = 10000
accepted_counts = []
for _ in range(n_trials):
    draft_t = [np.random.choice(vocab_size, p=d) for d in draft_distributions]
    acc = speculative_accept_reject(draft_distributions, target_distributions, draft_t)
    accepted_counts.append(len(acc))

print(f"\nOver {n_trials} trials:")
print(f"  Mean accepted tokens: {np.mean(accepted_counts):.2f} (out of max {K+1})")
print(f"  Acceptance rate: {(np.mean(accepted_counts)-1)/K*100:.1f}%")

## 3. Why This Works: The Math

**Draft phase:** K forward passes through the small model
- Each pass is fast (~10x cheaper than target if draft is 10x smaller)
- Total draft cost: K x t_draft

**Verify phase:** 1 forward pass through the large target model on K tokens
- This is like prefill: processing K tokens in parallel
- Cost: ~t_target (barely more expensive than generating 1 token, if K is small)

**Expected tokens per step:** depends on acceptance rate alpha
- If alpha = probability each draft token is accepted
- Expected accepted = (1 - alpha^(K+1)) / (1 - alpha) for geometric distribution
- At alpha=0.8, K=5: expected ~3.4 tokens per step

**Speedup** = expected_tokens / (K * t_draft + t_target) vs 1 / t_target

In [ ]:
# --- YOUR CODE: calculate expected tokens per step and speedup ---

def expected_tokens(alpha, K):
    """Expected number of accepted tokens given acceptance rate alpha and K draft tokens."""
    # --- YOUR CODE ---
    # Hint: E[tokens] = sum_{i=0}^{K} alpha^i * (1-alpha) * (i+1) + alpha^(K+1) * (K+1)
    # Simplified: (1 - alpha^(K+1)) / (1 - alpha) if alpha < 1
    if alpha >= 1.0:
        return K + 1
    return None  # YOUR CODE: (1 - alpha**(K+1)) / (1 - alpha)

def speculative_speedup(alpha, K, draft_cost_ratio=0.1):
    """Speedup of speculative decoding vs standard autoregressive.

    draft_cost_ratio: t_draft / t_target (e.g., 0.1 means draft is 10x cheaper)
    """
    # Standard: 1 token per t_target
    # Speculative: expected_tokens per (K * t_draft + t_target)
    e_tokens = expected_tokens(alpha, K)
    spec_time = K * draft_cost_ratio + 1.0  # in units of t_target
    # --- YOUR CODE ---
    # Hint: speedup = e_tokens / spec_time
    return None  # YOUR CODE

# Plot throughput vs acceptance rate
alphas = np.linspace(0.01, 0.99, 100)
Ks = [3, 5, 7, 10]
draft_ratio = 0.1  # draft is 10x cheaper

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for K_val in Ks:
    e_toks = [expected_tokens(a, K_val) for a in alphas]
    speedups = [speculative_speedup(a, K_val, draft_ratio) for a in alphas]
    ax1.plot(alphas, e_toks, label=f'K={K_val}', linewidth=2)
    ax2.plot(alphas, speedups, label=f'K={K_val}', linewidth=2)

ax1.set_xlabel('Acceptance Rate (alpha)', fontsize=12)
ax1.set_ylabel('Expected Tokens per Step', fontsize=12)
ax1.set_title('Expected Tokens vs Acceptance Rate', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='No speculation')

ax2.set_xlabel('Acceptance Rate (alpha)', fontsize=12)
ax2.set_ylabel('Speedup vs Autoregressive', fontsize=12)
ax2.set_title(f'Speedup (draft cost = {draft_ratio:.0%} of target)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Print key data points
print(f"\nKey data points (draft cost = {draft_ratio:.0%} of target):")
print(f"{'Alpha':>8} {'K':>4} {'E[tokens]':>12} {'Speedup':>10}")
print("="*38)
for a in [0.5, 0.7, 0.8, 0.9, 0.95]:
    for k in [5, 7]:
        et = expected_tokens(a, k)
        sp = speculative_speedup(a, k, draft_ratio)
        print(f"{a:>8.2f} {k:>4} {et:>12.2f} {sp:>10.2f}x")

## 4. Implementing Speculative Decoding

Full implementation with LLaMA-3.2-1B as the target model.

For the draft model, we use a simpler approach: a small lookup-based n-gram model or a tiny transformer.
In practice, you'd use a distilled version of the target or a purpose-trained draft model.

The key implementation challenge: the target model must process K tokens and produce K+1 probability
distributions in a single forward pass.

In [ ]:
# --- YOUR CODE: implement speculative decoding with a real or simulated model ---

# Option 1: With real models (uncomment if you have the models)
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
#
# target_model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.2-1B", torch_dtype=torch.float16).cuda().eval()
# # For draft, use a much smaller model
# draft_model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.2-1B", torch_dtype=torch.float16).cuda().eval()  # ideally a smaller model
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

# Option 2: Simulation to understand the algorithm
class SimulatedModel:
    """Simulated LM that generates from a fixed distribution with temperature."""
    def __init__(self, vocab_size, temperature=1.0, name="model"):
        self.vocab_size = vocab_size
        self.temperature = temperature
        self.name = name
        # Fixed "learned" distribution (like a unigram model)
        self.base_logits = np.random.randn(vocab_size)

    def get_probs(self, context):
        """Get next-token probabilities given context."""
        # In real model: full forward pass through transformer
        # Here: context-dependent perturbation of base distribution
        logits = self.base_logits + np.random.randn(self.vocab_size) * 0.3
        logits = logits / self.temperature
        probs = np.exp(logits - logits.max())
        return probs / probs.sum()

    def generate_token(self, context):
        probs = self.get_probs(context)
        return np.random.choice(self.vocab_size, p=probs), probs


def speculative_decode_step(target, draft, context, K):
    """
    One step of speculative decoding.

    Returns: (new_tokens, n_draft_calls, n_target_calls)
    """
    # Phase 1: Draft generates K candidates
    draft_tokens = []
    draft_probs_list = []
    ctx = list(context)
    for _ in range(K):
        token, probs = draft.generate_token(ctx)
        draft_tokens.append(token)
        draft_probs_list.append(probs)
        ctx.append(token)

    # Phase 2: Target verifies all K tokens in one pass
    # --- YOUR CODE: get target probabilities for all K+1 positions ---
    # Hint: in a real model, this is ONE forward pass on the K draft tokens
    target_probs_list = []
    ctx = list(context)
    for i in range(K + 1):
        probs = target.get_probs(ctx)
        target_probs_list.append(probs)
        if i < K:
            ctx.append(draft_tokens[i])

    # Phase 3: Accept/reject
    accepted = speculative_accept_reject(draft_probs_list, target_probs_list, draft_tokens)

    return accepted, K, 1  # K draft calls, 1 target call


# Run speculative decoding
np.random.seed(42)
vocab_size = 1000
K = 5

# Target and draft share similar distributions (simulating a good draft model)
target = SimulatedModel(vocab_size, temperature=1.0, name="target")
draft = SimulatedModel(vocab_size, temperature=1.0, name="draft")
# Make draft similar to target (good draft model)
draft.base_logits = target.base_logits + np.random.randn(vocab_size) * 0.5

context = [0, 1, 2]  # initial context
total_tokens = 0
total_target_calls = 0
total_draft_calls = 0
n_steps = 200

for step in range(n_steps):
    tokens, d_calls, t_calls = speculative_decode_step(target, draft, context, K)
    context.extend(tokens)
    total_tokens += len(tokens)
    total_target_calls += t_calls
    total_draft_calls += d_calls

avg_accepted = total_tokens / n_steps
print(f"Speculative Decoding Results (K={K}):")
print(f"  Total tokens generated: {total_tokens}")
print(f"  Total target forward passes: {total_target_calls}")
print(f"  Total draft forward passes: {total_draft_calls}")
print(f"  Average tokens per target call: {avg_accepted:.2f}")
print(f"  Acceptance rate: {(avg_accepted - 1) / K * 100:.1f}%")
print(f"\n  Standard decode would need {total_tokens} target calls")
print(f"  Speculative used only {total_target_calls} target calls")
print(f"  Reduction: {(1 - total_target_calls / total_tokens) * 100:.1f}% fewer target calls")

## 5. Choosing the Draft Model

The draft model tradeoff:
- **Bigger draft** = higher acceptance rate (closer to target) but **slower drafting**
- **Smaller draft** = lower acceptance rate but **faster drafting**

The optimal draft model balances:
- Draft cost per token (proportional to draft model size)
- Acceptance rate (how well draft approximates target)

Rule of thumb: draft model should be ~5-20x smaller than target.

In [ ]:
# --- YOUR CODE: vary draft model quality and measure end-to-end throughput ---

def simulate_speculative_throughput(alpha, K, draft_cost_ratio, target_time_ms):
    """Calculate effective tokens/sec for speculative decoding."""
    e_tokens = expected_tokens(alpha, K)
    # Total time for one speculative step
    step_time_ms = K * draft_cost_ratio * target_time_ms + target_time_ms
    tokens_per_sec = e_tokens / step_time_ms * 1000
    return tokens_per_sec, e_tokens

# Target model: LLaMA-3.2-1B, ~6.7 ms/token at batch=1
target_time_ms = 6.7

# Different draft model configurations
draft_configs = [
    {"name": "Tiny (50x smaller)",   "cost_ratio": 0.02, "alpha": 0.50},
    {"name": "Small (20x smaller)",  "cost_ratio": 0.05, "alpha": 0.65},
    {"name": "Medium (10x smaller)", "cost_ratio": 0.10, "alpha": 0.78},
    {"name": "Large (5x smaller)",   "cost_ratio": 0.20, "alpha": 0.88},
    {"name": "XL (2x smaller)",      "cost_ratio": 0.50, "alpha": 0.93},
]

K_values = [3, 5, 7]
baseline_tps = 1000 / target_time_ms  # standard decode

print(f"Baseline (standard decode): {baseline_tps:.0f} tok/s")
print(f"\n{'Draft Model':<22} {'K':>3} {'Alpha':>7} {'E[tok]':>8} {'tok/s':>8} {'Speedup':>9}")
print("="*62)

best_speedup = 0
best_config = ""

for cfg in draft_configs:
    for K_val in K_values:
        tps, e_tok = simulate_speculative_throughput(
            cfg["alpha"], K_val, cfg["cost_ratio"], target_time_ms)
        speedup = tps / baseline_tps
        marker = " <-- best" if speedup > best_speedup else ""
        if speedup > best_speedup:
            best_speedup = speedup
            best_config = f"{cfg['name']}, K={K_val}"
        print(f"{cfg['name']:<22} {K_val:>3} {cfg['alpha']:>7.2f} {e_tok:>8.1f} {tps:>8.0f} {speedup:>8.2f}x{marker}")
    print()

print(f"Best configuration: {best_config} -> {best_speedup:.2f}x speedup")

## 6. Medusa: Multi-Head Draft Prediction

Instead of a separate draft model, **Medusa** adds lightweight prediction heads to the target model itself.

Architecture:
- Target model produces hidden states as usual
- K extra "Medusa heads" (small MLPs) predict future tokens at positions +1, +2, ..., +K
- Each head: `hidden_state -> Linear -> SiLU -> Linear -> vocab_logits`

Advantages over separate draft model:
- No extra model to load/maintain
- Heads see target model's rich representations
- Very low overhead (~2-5% extra parameters)

Disadvantage:
- Head predictions get worse for further positions
- Needs fine-tuning on target model's hidden states

In [ ]:
# --- YOUR CODE: implement simple Medusa-style prediction heads ---

# Simulated Medusa head dimensions
hidden_size = d_model  # 2048
medusa_hidden = 1024   # smaller intermediate dim
vocab_size_sim = 32000 # LLaMA vocab
n_medusa_heads = 5     # predict 5 future positions

# --- YOUR CODE: calculate Medusa head parameter count ---
# Each head: Linear(hidden_size, medusa_hidden) + Linear(medusa_hidden, vocab_size)
params_per_head = None  # YOUR CODE: hidden_size * medusa_hidden + medusa_hidden * vocab_size_sim
total_medusa_params = None  # YOUR CODE: params_per_head * n_medusa_heads

# Compare to target model params
medusa_overhead = total_medusa_params / total_params * 100

print(f"Medusa Head Analysis")
print(f"  Hidden size: {hidden_size}")
print(f"  Medusa hidden dim: {medusa_hidden}")
print(f"  Vocab size: {vocab_size_sim}")
print(f"  Number of heads: {n_medusa_heads}")
print(f"\n  Params per head: {params_per_head / 1e6:.1f}M")
print(f"  Total Medusa params: {total_medusa_params / 1e6:.1f}M")
print(f"  Target model params: {total_params / 1e6:.0f}M")
print(f"  Overhead: {medusa_overhead:.1f}%")

# Extra memory bandwidth per decode step
medusa_bytes = total_medusa_params * 2  # FP16
extra_time_ms = medusa_bytes / (hbm_bandwidth * 1e9) * 1000
print(f"\n  Extra memory to load: {medusa_bytes / 1e6:.1f} MB")
print(f"  Extra time per step: {extra_time_ms:.3f} ms")
print(f"  vs target forward pass: {time_per_token_ms:.2f} ms")
print(f"  Overhead: {extra_time_ms / time_per_token_ms * 100:.1f}% of forward pass")

# Typical Medusa acceptance rates by position
positions = list(range(1, n_medusa_heads + 1))
typical_acceptance = [0.85, 0.70, 0.55, 0.40, 0.28]  # degrades with distance

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(positions, typical_acceptance, color='steelblue', alpha=0.8)
ax.set_xlabel('Prediction Position (+N tokens ahead)', fontsize=12)
ax.set_ylabel('Typical Acceptance Rate', fontsize=12)
ax.set_title('Medusa: Acceptance Rate by Prediction Position', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for i, (pos, rate) in enumerate(zip(positions, typical_acceptance)):
    ax.text(pos, rate + 0.02, f'{rate:.0%}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

# Expected tokens from Medusa with tree verification
# (simplified: product of acceptance rates)
expected_from_medusa = 1  # always get at least 1
for rate in typical_acceptance:
    expected_from_medusa += rate
print(f"\nExpected tokens per step (greedy, no tree): {expected_from_medusa:.2f}")

## 7. Eagle: Feature-Level Speculation

**Eagle** (Extrapolation Algorithm for Greater Language-model Efficiency) improves on Medusa by using
**feature-level information** from the target model's layers, not just the final hidden state.

Key insight: future tokens depend on features at multiple layers, not just the last one.

Architecture:
- Extract features from target model's intermediate layers
- A lightweight "draft head" processes these features to predict next tokens
- Achieves higher acceptance rates than Medusa because features carry richer information

Results (from Eagle paper):
- ~40% higher acceptance rate than Medusa on average
- 1.5-2x faster than standard decoding on LLaMA models
- Works especially well on reasoning/code tasks where patterns are more predictable

Why feature-level helps:
- The target model's intermediate layers already encode "planning" information
- Layer N-1 features often predict what layer N will produce
- Eagle exploits this temporal structure in the network

In [ ]:
# Eagle comparison (conceptual — no implementation, just analysis)

# Acceptance rates from papers
methods = ['Standard Draft\n(separate model)', 'Medusa\n(prediction heads)', 'Eagle\n(feature-level)']
acceptance_rates = [0.75, 0.70, 0.85]  # typical on LLaMA-2-Chat
avg_tokens_per_step = [expected_tokens(a, 5) for a in acceptance_rates]
speedups_over_base = [speculative_speedup(a, 5, cost) for a, cost in
                      zip(acceptance_rates, [0.10, 0.03, 0.05])]  # different overhead costs

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(methods, acceptance_rates, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8)
axes[0].set_ylabel('Acceptance Rate', fontsize=12)
axes[0].set_title('Acceptance Rate', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1)
for i, v in enumerate(acceptance_rates):
    axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', fontsize=11)

axes[1].bar(methods, avg_tokens_per_step, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8)
axes[1].set_ylabel('Expected Tokens per Step', fontsize=12)
axes[1].set_title('Tokens per Target Call (K=5)', fontsize=13, fontweight='bold')
for i, v in enumerate(avg_tokens_per_step):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=11)

axes[2].bar(methods, speedups_over_base, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8)
axes[2].set_ylabel('Speedup vs Standard Decode', fontsize=12)
axes[2].set_title('End-to-End Speedup', fontsize=13, fontweight='bold')
axes[2].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
for i, v in enumerate(speedups_over_base):
    axes[2].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 8. Tree-Based Speculation

Instead of a single draft sequence, generate a **tree** of candidates.

At each draft position, keep the **top-k** most likely tokens, creating a branching tree.
Verify **all branches** in one target forward pass using a **tree attention mask**.

Example tree (top-2 at each level, 3 levels):
```
         root
        /    \
      "The"  "A"
      / \    / \
  "cat" "dog" "big" "small"
```

Tree attention mask: each node can attend to its ancestors (path from root), but NOT to siblings or cousins.
This is different from the standard causal mask — it's a tree-structured mask.

In [ ]:
# --- YOUR CODE: implement tree verification with attention mask ---

def build_tree_attention_mask(tree_structure):
    """
    Build attention mask for tree-based speculation.

    tree_structure: list of (token_id, parent_index) pairs.
                    parent_index=-1 for root.

    Returns: (n_nodes, n_nodes) boolean mask where mask[i][j]=True means
             node i can attend to node j.
    """
    n_nodes = len(tree_structure)
    mask = np.zeros((n_nodes, n_nodes), dtype=bool)

    # --- YOUR CODE: each node can attend to itself and all ancestors ---
    # Hint: for each node, walk up the parent chain and set mask[i][ancestor] = True
    for i in range(n_nodes):
        # Self-attention
        mask[i][i] = True
        # Walk up parent chain
        current = tree_structure[i][1]  # parent index
        while current >= 0:
            mask[i][current] = True  # YOUR CODE
            current = tree_structure[current][1]  # YOUR CODE: grandparent

    return mask


# Build example tree: root -> 2 children -> 4 grandchildren
# (token_id, parent_index)
tree = [
    (0,  -1),  # 0: root (prompt token)
    (10,  0),  # 1: "The" (child of root)
    (20,  0),  # 2: "A"   (child of root)
    (30,  1),  # 3: "cat" (child of "The")
    (40,  1),  # 4: "dog" (child of "The")
    (50,  2),  # 5: "big" (child of "A")
    (60,  2),  # 6: "small" (child of "A")
]

mask = build_tree_attention_mask(tree)

print("Tree structure:")
labels = ['root', 'The', 'A', 'cat', 'dog', 'big', 'small']
for i, (tok, parent) in enumerate(tree):
    parent_name = labels[parent] if parent >= 0 else 'None'
    print(f"  [{i}] {labels[i]:<8} parent={parent_name}")

print(f"\nTree attention mask:")
print(f"{'':>8}", end='')
for l in labels:
    print(f"{l:>7}", end='')
print()

for i, label in enumerate(labels):
    print(f"{label:>8}", end='')
    for j in range(len(labels)):
        print(f"{'  T':>7}" if mask[i][j] else f"{'  .':>7}", end='')
    print()

# Compare with standard causal mask
print(f"\nStandard causal mask would have {sum(range(1, len(tree)+1))} True entries")
print(f"Tree mask has {mask.sum()} True entries")
print(f"Tree mask is sparser -> less computation, but explores more paths")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(mask, cmap='Blues', aspect='auto')
ax1.set_xticks(range(len(labels)))
ax1.set_yticks(range(len(labels)))
ax1.set_xticklabels(labels, rotation=45, fontsize=10)
ax1.set_yticklabels(labels, fontsize=10)
ax1.set_title('Tree Attention Mask', fontsize=14, fontweight='bold')
ax1.set_xlabel('Key (attends to)')
ax1.set_ylabel('Query')

causal_mask = np.tril(np.ones((len(tree), len(tree)), dtype=bool))
ax2.imshow(causal_mask, cmap='Reds', aspect='auto')
ax2.set_xticks(range(len(labels)))
ax2.set_yticks(range(len(labels)))
ax2.set_xticklabels(labels, rotation=45, fontsize=10)
ax2.set_yticklabels(labels, fontsize=10)
ax2.set_title('Standard Causal Mask', fontsize=14, fontweight='bold')
ax2.set_xlabel('Key (attends to)')
ax2.set_ylabel('Query')

plt.tight_layout()
plt.show()

## 9. Benchmark: Comparing Methods

Compare decode speeds across different optimization combinations:
- (a) Standard autoregressive decoding
- (b) Speculative decoding
- (c) Speculative + quantization
- (d) Speculative + continuous batching

In [ ]:
# --- YOUR CODE: compare decode methods ---

# LLaMA-3.2-1B on A100, estimated performance
methods = {
    'Standard FP16': {
        'tok_per_sec': 150,
        'latency_ms': 6.7,
        'description': 'Batch=1, FP16, no tricks',
    },
    'Speculative (K=5)': {
        # --- YOUR CODE ---
        # Hint: ~2.5x speedup with good draft model (alpha=0.8)
        'tok_per_sec': None,  # YOUR CODE: ~375
        'latency_ms': None,   # YOUR CODE: ~2.7
        'description': 'Batch=1, FP16, draft model 10x smaller',
    },
    'Spec + INT8 Quant': {
        # --- YOUR CODE ---
        # Hint: quantization gives ~1.8x on top of speculation
        'tok_per_sec': None,  # YOUR CODE: ~675
        'latency_ms': None,   # YOUR CODE: ~1.5
        'description': 'Batch=1, INT8 weights, speculative K=5',
    },
    'Spec + Cont. Batch': {
        # --- YOUR CODE ---
        # Hint: batching gives ~10x throughput but latency stays similar
        'tok_per_sec': None,  # YOUR CODE: ~3750 (throughput)
        'latency_ms': None,   # YOUR CODE: ~4.3 (per-request latency increases)
        'description': 'Batch=16, FP16, speculative K=5',
    },
}

print(f"{'Method':<25} {'tok/s':>10} {'ms/tok':>10} {'Description'}")
print("="*80)
for name, data in methods.items():
    print(f"{name:<25} {data['tok_per_sec']:>10.0f} {data['latency_ms']:>10.1f} {data['description']}")

# Bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = list(methods.keys())
tps = [methods[n]['tok_per_sec'] for n in names]
lats = [methods[n]['latency_ms'] for n in names]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

ax1.barh(names, tps, color=colors, alpha=0.8)
ax1.set_xlabel('Throughput (tokens/sec)', fontsize=12)
ax1.set_title('Throughput Comparison', fontsize=14, fontweight='bold')
for i, v in enumerate(tps):
    ax1.text(v + 50, i, f'{v:.0f}', va='center', fontsize=11)

ax2.barh(names, lats, color=colors, alpha=0.8)
ax2.set_xlabel('Latency (ms/token)', fontsize=12)
ax2.set_title('Per-Token Latency Comparison', fontsize=14, fontweight='bold')
for i, v in enumerate(lats):
    ax2.text(v + 0.1, i, f'{v:.1f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

## 10. The Full Stack: Combining Everything

Chapters 1-10 form a complete optimization stack for LLM inference. Here's the summary:

| Chapter | Topic | Key Optimization |
|---|---|---|
| Ch1 | GPU Architecture & Roofline | Understand the hardware limits |
| Ch2 | Transformer Profiling | Know where time goes |
| Ch3 | KV Cache & GQA/MLA | Reduce cache memory |
| Ch4 | Flash Attention | Fuse attention, cut HBM traffic |
| Ch5 | Memory Management | Efficient activation/gradient memory |
| Ch6 | Kernel Fusion & torch.compile | Fuse elementwise ops |
| Ch7 | Quantization | Shrink weights and activations |
| Ch8 | Advanced Quantization | GPTQ, AWQ, mixed-precision |
| Ch9 | Continuous Batching & PagedAttention | Serve many users efficiently |
| Ch10 | Speculative Decoding | Generate multiple tokens per pass |

Calculate the combined speedup from applying the full stack.

In [ ]:
# --- YOUR CODE: calculate end-to-end improvement from the full stack ---

# Starting point: naive batch=1 FP16 decode on LLaMA-3.2-1B
baseline_ms_per_token = 6.7
baseline_tps = 1000 / baseline_ms_per_token  # ~149 tok/s

# Each optimization contributes a multiplicative factor
optimizations = [
    # (name, chapter, throughput_factor, latency_factor, description)
    # throughput_factor: how much total throughput improves
    # latency_factor: how much per-request latency improves (>1 = faster)

    # --- YOUR CODE: fill in estimated factors ---
    # Hint: these should be reasonable estimates based on what you learned
    ('INT8 Quantization', 'Ch7', None, None,    # YOUR CODE: throughput 1.8, latency 1.8
     'Halve weight bytes -> nearly 2x decode speed'),
    ('Flash Attention', 'Ch4', None, None,       # YOUR CODE: throughput 1.3, latency 1.3
     'Fused attention -> less HBM traffic'),
    ('Kernel Fusion', 'Ch6', None, None,         # YOUR CODE: throughput 1.2, latency 1.2
     'Fuse RMSNorm/SiLU/residual ops'),
    ('Continuous Batching', 'Ch9', None, None,   # YOUR CODE: throughput 12.0, latency 0.8 (latency slightly worse)
     'Batch=16 effective -> amortize weight loads'),
    ('Speculative Decoding', 'Ch10', None, None, # YOUR CODE: throughput 2.5, latency 2.5
     'K=5 tokens per target call, alpha=0.8'),
]

print(f"Full Stack Optimization Summary (LLaMA-3.2-1B on A100)")
print(f"{'='*75}")
print(f"{'Optimization':<25} {'Ch':>4} {'Throughput':>12} {'Latency':>12} {'Notes'}")
print(f"{'-'*75}")

cum_throughput = baseline_tps
cum_latency = baseline_ms_per_token

print(f"{'Baseline (naive FP16)':<25} {'--':>4} {baseline_tps:>9.0f} tok/s {baseline_ms_per_token:>8.1f} ms/tok")

for name, ch, tp_factor, lat_factor, desc in optimizations:
    cum_throughput *= tp_factor
    cum_latency /= lat_factor
    print(f"{'+ ' + name:<25} {ch:>4} {cum_throughput:>9.0f} tok/s {cum_latency:>8.2f} ms/tok  ({desc})")

total_tp_speedup = cum_throughput / baseline_tps
total_lat_speedup = baseline_ms_per_token / cum_latency

print(f"{'-'*75}")
print(f"{'TOTAL':<25} {'':>4} {cum_throughput:>9.0f} tok/s {cum_latency:>8.2f} ms/tok")
print(f"\nThroughput improvement: {total_tp_speedup:.0f}x ({baseline_tps:.0f} -> {cum_throughput:.0f} tok/s)")
print(f"Latency improvement: {total_lat_speedup:.1f}x ({baseline_ms_per_token:.1f} -> {cum_latency:.2f} ms/tok)")

# Note on throughput vs latency
print(f"\nNote: Throughput and latency don't scale the same way.")
print(f"Batching massively helps throughput but slightly hurts per-request latency.")
print(f"Speculative decoding helps BOTH throughput and latency.")
print(f"Quantization helps BOTH by reducing bytes transferred per token.")

# Final visualization
fig, ax = plt.subplots(figsize=(10, 6))

labels = ['Baseline'] + [name for name, _, _, _, _ in optimizations]
cumulative_tps = [baseline_tps]
running = baseline_tps
for _, _, tp, _, _ in optimizations:
    running *= tp
    cumulative_tps.append(running)

bars = ax.bar(range(len(labels)), cumulative_tps,
              color=['gray'] + ['steelblue'] * len(optimizations), alpha=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Throughput (tokens/sec)', fontsize=12)
ax.set_title('Cumulative Throughput: Full Optimization Stack', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, cumulative_tps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## Revision Notes

*Fill this in after running all sections.*

---

**Speculative decoding core idea:**
- Draft model generates ___ candidate tokens
- Target model verifies in ___ forward pass(es)
- Quality guarantee: ___

**Accept/reject rule:**
- Accept probability: min(1, ___)
- On rejection, sample from: ___

**Expected tokens formula:**
- E[tokens] = ___
- At alpha=0.8, K=5: ___ tokens per step

**Draft model tradeoff:**
- Bigger draft: ___ acceptance, ___ speed
- Optimal draft size: ~___x smaller than target

**Medusa vs Eagle vs separate draft:**
- Medusa: ___
- Eagle: ___
- Separate draft: ___

**Tree attention mask difference from causal:**



**Full stack speedup (Chapters 1-10):**
- Throughput: ___x
- Latency: ___x

**What surprised me:**



---
*End of LLM GPU Performance series*